In [1]:
import os
%pwd

'd:\\web Development using python\\PROJECTS\\kidney_disease_classification\\research'

In [2]:
os.chdir("../")

In [3]:
%pwd

'd:\\web Development using python\\PROJECTS\\kidney_disease_classification'

In [4]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size :list

In [5]:
from kidney_disease_classifier.constants import *
from kidney_disease_classifier.utils.common import get_size, read_yaml , create_directories
from kidney_disease_classifier import logger
import tensorflow as tf

In [6]:

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:
        training_config = self.config.training

        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "kidney-ct-scan-image")

        create_directories([training_config.root_dir])

        training_config = TrainingConfig(
            root_dir=Path(training_config.root_dir),
            trained_model_path=Path(training_config.trained_model_path),
            updated_base_model_path=Path(self.config.prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=self.params.EPOCHS,
            params_batch_size=self.params.BATCH_SIZE,
            params_is_augmentation=self.params.AUGMENTATION,
            params_image_size=self.params.IMAGE_SIZE
        )

        return training_config



In [7]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [8]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        logger.info("Getting the base model")
        self.model = tf.keras.models.load_model(self.config.updated_base_model_path)
        logger.info("Got the base model")


    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )


    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)




    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )


In [9]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    logger.info(f"Training config: {training_config}")
    trainer = Training(config=training_config)
    trainer.get_base_model()
    trainer.train_valid_generator()
    trainer.train()
except Exception as e:
    logger.exception(e)
    raise e

[2026-04-06 12:25:58,162: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-06 12:25:58,167: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-06 12:25:58,172: INFO: common: created directory at: artifacts]
[2026-04-06 12:25:58,176: INFO: common: created directory at: artifacts/training]
[2026-04-06 12:25:58,178: INFO: 2295746223: Training config: TrainingConfig(root_dir=WindowsPath('artifacts/training'), trained_model_path=WindowsPath('artifacts/training/model.h5'), updated_base_model_path=WindowsPath('artifacts/prepare_base_model/base_model_updated.h5'), training_data=WindowsPath('artifacts/data_ingestion/kidney-ct-scan-image'), params_epochs=1, params_batch_size=16, params_is_augmentation=True, params_image_size=BoxList([224, 224, 3]))]
[2026-04-06 12:25:58,181: INFO: 3912924164: Getting the base model]
[2026-04-06 12:25:59,045: INFO: 3912924164: Got the base model]
Found 93 images belonging to 2 classes.
Found 372 images belonging to 2 class